# Cheap Regression Prediction Model

In [ ]:
import re
from imbalance_sa.paths import PROJECT_ROOT     # Specifies the root, distinguishing between my work and personal laptop
from imbalance_sa.data.catalog import data      # imports all CSV files as Pandas DataFrames

import pandas as pd
import numpy as np
import datetime as dt
import plotly.graph_objects as go
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

## 1. Forecast runs
One model per auction, each run 30 minutes before that auction's gate closure and using only data available at that moment (all times Irish local time).

| Run | Run time | Gate closure | NIV predicted for (trading day D) |
|---|---|---|---|
| DA | 10:30 D-1 | 11:00 D-1 | 23:00 D-1 to 23:00 D (48 periods) |
| IDA1 | 17:00 D-1 | 17:30 D-1 | 23:00 D-1 to 23:00 D (48 periods) |
| IDA2 | 07:30 D | 08:00 D | 11:00 D to 23:00 D (24 periods, the IDA2 delivery window) |
| IDA3 | 13:30 D | 14:00 D | 17:00 D to 23:00 D (12 periods, the IDA3 delivery window) |

**No-leakage rules**
1. **Auction results** (prices, cleared volumes) are only used from the *next* run onwards. GB IDA1/IDA2 are coupled with the Irish auctions, so they follow the same rule.
   **GB day-ahead results** (D-1, UTC): EPEX hourly (09:20) is used from DA; Nord Pool N2EX (09:50, after the 10:30 IST DA run in summer) and the EPEX half-hourly auction (results from 15:45, before the 17:30 SEM IDA1 gate) are used from IDA1.
2. **Forecasts** carry their issue time (`tick_utc`). Any forecast value issued at or after the run time is removed for that run.
3. **Daily / weekly series** (FX, NBP gas) are treated as published one full period after their timestamp, and each run takes the latest value published before its run time.
4. The same-period imbalance price is never used. Section 3 checks rules 1–3 on the data for every period.

In [ ]:
START = "2018-10-01"                                       # data start

RUNS = ["DA", "IDA1", "IDA2", "IDA3"]

# Run time of each model relative to trading day D (Irish time): (day offset, clock time)
RUN_TIME = {"DA": (-1, "10:30"), "IDA1": (-1, "17:00"), "IDA2": (0, "07:30"), "IDA3": (0, "13:30")}

# Delivery window of each run, in hours after the trading-day start (23:00 D-1): [start, end)
WINDOW = {"DA": (0, 24), "IDA1": (0, 24), "IDA2": (12, 24), "IDA3": (18, 24)}

TARGET = "ie_niv_actual_mw"
EXCLUDE = [TARGET, "ie_pimb_actual_eur"]       # same-period imbalance price is only known after delivery
META_COLS = ["trading_day", "auction", "cutoff_utc", "tick_utc"]


def run_datetime(trading_day, run: str) -> pd.Series:
    """Run time (tz-aware Irish time) for each trading day."""
    days, clock = RUN_TIME[run]
    td = pd.to_datetime(pd.Series(trading_day))
    return (td + pd.Timedelta(days=days) + pd.Timedelta(clock + ":00")).dt.tz_localize("Europe/Dublin")


# GB day-ahead results, published at a fixed time on D-1 (UTC): (first run allowed, publication time).
# Each run also re-checks publication against its own run time per trading day (see run_frame).
PUBLISHED_UTC = {
    "gb_da_actual_epex_gbp":    ("DA",   "09:20"),   # EPEX Spot GB DA hourly
    "gb_da_actual_n2ex_gbp":    ("IDA1", "09:50"),   # Nord Pool N2EX GB DA hourly: after the 10:30 IST DA run in summer
    "gb_da_hh_actual_epex_gbp": ("IDA1", "15:45"),   # EPEX Spot GB DA half-hourly: before the 17:30 SEM IDA1 gate
}


def first_run(name: str) -> str | None:
    """First run at which a dataset is known (None = never known before IDA3)."""
    if name in PUBLISHED_UTC:                                    # GB DA results (hourly and half-hourly)
        return PUBLISHED_UTC[name][0]
    m = re.match(r"(?:ie|gb)_(da|ida1|ida2|ida3)_(mcv|actual)", name)   # auction results: known after that
    if m:                                                               # auction (GB IDAs are coupled with IE)
        i = RUNS.index(m.group(1).upper()) + 1
        return RUNS[i] if i < len(RUNS) else None
    m = re.search(r"_(da|ida1|ida2|ida3)$", name)                # forecasts made for a specific auction
    if m:
        return m.group(1).upper()
    return "DA"   # price forecasts and lagged daily/weekly data (checked against run time below)


def step_of(s) -> pd.Timedelta:
    return pd.Timedelta(s.index.to_series().diff().mode().iloc[0])


names = [n for n in data.list().index if n not in EXCLUDE]
SLOW = [n for n in names if step_of(data[n]) >= pd.Timedelta(days=1)]     # daily / weekly series
FAST = [n for n in names if n not in SLOW]

features_for = {
    run: [n for n in names if first_run(n) is not None and RUNS.index(first_run(n)) <= RUNS.index(run)]
    for run in RUNS
}
pd.DataFrame({"datasets available": {r: len(f) for r, f in features_for.items()},
              "run time": {r: f"{c} D{d:+d}" if d else f"{c} D" for r, (d, c) in RUN_TIME.items()}})

## 2. Build the dataset
All half-hourly and hourly series on one half-hourly grid in Irish time. Forecast values issued at or after the run time of their first run are removed.

In [ ]:
USED_TICK = {}   # issue time of every forecast value actually used, for the leakage check


def to_grid(s: pd.Series, step: pd.Timedelta) -> pd.Series:
    n_slots = max(int(step // pd.Timedelta(minutes=30)) - 1, 0)          # hourly -> fill the :30 slot
    return s.resample("30min").ffill(limit=n_slots) if n_slots else s.resample("30min").last()


def fast_feature(name: str) -> pd.Series:
    """Half-hourly series for one dataset, with forecasts issued too late for their run removed."""
    df = data[name]
    s = df.drop(columns=META_COLS, errors="ignore").iloc[:, 0]
    step = step_of(s)
    if "tick_utc" in df.columns:
        too_late = df["tick_utc"].values >= run_datetime(df["trading_day"], first_run(name)).values
        s = s.mask(too_late)
        USED_TICK[name] = to_grid(df["tick_utc"].where(s.notna()), step)
    return to_grid(s, step)


full = pd.DataFrame({n: fast_feature(n) for n in FAST})
full = full.join(data[TARGET]["niv_mw"], how="right")
full = full[full["niv_mw"].notna()].loc[START:]

# Trading day D runs 23:00 D-1 to 23:00 D (Irish time): periods from 23:00 belong to the next day
local_date = full.index.normalize().tz_localize(None)
full["trading_day"] = local_date + pd.to_timedelta((full.index.hour == 23).astype("int64"), unit="D")

# Position in the trading day by the Irish clock: 23:00 -> 0, 11:00 -> 12, 17:00 -> 18
full["hours_into_day"] = (full.index.hour + full.index.minute / 60 + 1) % 24

# Calendar features (Irish local time)
full["hour"] = full.index.hour + full.index.minute / 60
full["dayofweek"] = full.index.dayofweek
full["month"] = full.index.month
CALENDAR = ["hour", "dayofweek", "month"]


def published_time(name: str, frame: pd.DataFrame) -> pd.Series:
    """Publication time (tz-aware) of a PUBLISHED_UTC dataset for each period's trading day (auction on D-1)."""
    day_before = pd.to_datetime(frame["trading_day"]) - pd.Timedelta(days=1)
    t = (day_before + pd.Timedelta(PUBLISHED_UTC[name][1] + ":00")).dt.tz_localize("UTC")
    return pd.Series(t.values, index=frame.index)


def run_frame(run: str) -> pd.DataFrame:
    """Rows and features for one run, exactly as they would have been known at the run time."""
    lo, hi = WINDOW[run]
    d = full[(full["hours_into_day"] >= lo) & (full["hours_into_day"] < hi)].copy()
    run_dt = pd.DatetimeIndex(run_datetime(d["trading_day"].values, run))

    # Results published at a fixed time on D-1: blank them on days the run starts before publication
    for n in PUBLISHED_UTC:
        if n in features_for[run]:
            late = published_time(n, d).values >= run_dt.values
            d.loc[late, n] = np.nan

    # Daily/weekly series: each value counts as published one full period after its timestamp; use the
    # latest value published before the run time for the whole trading day
    for n in SLOW:
        if n in features_for[run]:
            s = data[n].drop(columns=META_COLS, errors="ignore").iloc[:, 0].dropna()
            published = s.copy()
            published.index = s.index + step_of(s)
            d[n] = published.reindex(run_dt, method="ffill").values
    return d


print(f"{len(full):,} half-hours, {full['trading_day'].nunique():,} trading days")

## 3. Leakage check
Built from exactly the frames the models train and test on. Every forecast value must have been issued before its run started, and every published result or daily/weekly value must have been published before it. All counts must be zero.

In [ ]:
checks = {}
for run in RUNS:
    d = run_frame(run)
    rd = run_datetime(d["trading_day"].values, run).values           # run time per period
    row = {"forecasts issued after run time": 0,
           "GB DA results published after run time": 0,
           "daily/weekly values published after run time": 0}

    for n in features_for[run]:
        used = d[n].notna().values
        if n in USED_TICK:                                            # forecasts: issue time of the value used
            tick = USED_TICK[n].reindex(d.index).values
            row["forecasts issued after run time"] += int(((tick >= rd) & used).sum())
        if n in PUBLISHED_UTC:                                        # fixed-time results
            pub = published_time(n, d).values
            row["GB DA results published after run time"] += int(((pub >= rd) & used).sum())
        if n in SLOW:                                                 # daily/weekly: publication of value used
            s = data[n].iloc[:, -1].dropna()
            pub_t = pd.Series(s.index + step_of(s), index=s.index + step_of(s))
            pub = pub_t.reindex(pd.DatetimeIndex(run_datetime(d["trading_day"].values, run)), method="ffill").values
            row["daily/weekly values published after run time"] += int(((pub >= rd) & used).sum())
    checks[run] = row

leak_check = pd.DataFrame(checks).T
assert leak_check.values.sum() == 0, "Data leak detected"
leak_check


## 4. Train / test split (80 / 20 by trading day)
The same split date is used for every run so the four models are tested on the same days.

In [ ]:
days = np.sort(full["trading_day"].unique())
split_day = days[int(len(days) * 0.8)]
print(f"Train: trading days before {pd.Timestamp(split_day):%Y-%m-%d}   Test: from {pd.Timestamp(split_day):%Y-%m-%d}")

## 5. Train one XGBoost model per run

In [ ]:
models, predictions, rows = {}, {}, {}

for run in RUNS:
    d = run_frame(run)
    cols = CALENDAR + features_for[run]

    train, test = d[d["trading_day"] < split_day], d[d["trading_day"] >= split_day]
    model = XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42)
    model.fit(train[cols], train["niv_mw"])

    pred = pd.Series(model.predict(test[cols]), index=test.index)
    base = np.full(len(test), train["niv_mw"].mean())
    models[run], predictions[run] = model, pred
    rows[run] = {
        "features": len(cols),
        "test periods": len(test),
        "MAE (MW)": mean_absolute_error(test["niv_mw"], pred),
        "RMSE (MW)": mean_squared_error(test["niv_mw"], pred) ** 0.5,
        "R²": r2_score(test["niv_mw"], pred),
        "Baseline MAE (MW)": mean_absolute_error(test["niv_mw"], base),
    }

results = pd.DataFrame(rows).T.round(3)
results

## 6. Results

In [ ]:
# Top 10 features for each run
pd.DataFrame({
    run: pd.Series(m.feature_importances_, index=m.feature_names_in_).nlargest(10).index.tolist()
    for run, m in models.items()
})

In [ ]:
# Whole test set: actual NIV vs each run's forecast (drag the slider or zoom to inspect)
actual = full.loc[full["trading_day"] >= split_day, "niv_mw"]

fig = go.Figure()
fig.add_trace(go.Scattergl(x=actual.index, y=actual, name="Actual NIV", line=dict(color="black", width=1.5)))
for run, pred in predictions.items():
    pred = pred.reindex(actual.index)          # periods outside the run's window become NaN -> line breaks
    fig.add_trace(go.Scattergl(x=pred.index, y=pred, name=f"{run} forecast", line=dict(width=1), connectgaps=False))
fig.update_layout(
    title=f"NIV forecasts by run: test set from {pd.Timestamp(split_day):%d %b %Y}",
    yaxis_title="NIV (MW)", xaxis_title="Delivery period (Irish time)", height=550,
    xaxis=dict(rangeslider=dict(visible=True)), hovermode="x unified",
)
fig.show()

In [ ]:
# Forecast statistics for each run on the test set
from sklearn.metrics import median_absolute_error


def forecast_stats(y_true: pd.Series, y_pred: pd.Series, y_base: float) -> dict:
    """Standard point-forecast metrics, plus NIV direction accuracy and tail error."""
    err = y_pred - y_true
    tail = y_true.abs() >= y_true.abs().quantile(0.95)          # the 5% largest imbalances (either sign)
    mae, mae_base = err.abs().mean(), (y_true - y_base).abs().mean()
    return {
        "periods": len(y_true),
        "MAE (MW)": mae,
        "RMSE (MW)": np.sqrt((err ** 2).mean()),
        "MedAE (MW)": median_absolute_error(y_true, y_pred),
        "Bias (MW)": err.mean(),                                   # + = over-forecasts NIV
        "P90 abs error (MW)": err.abs().quantile(0.90),
        "R²": r2_score(y_true, y_pred),
        "Correlation": y_true.corr(y_pred),
        "Direction hit rate": (np.sign(y_true) == np.sign(y_pred)).mean(),   # short vs long called correctly
        "Tail MAE (MW, top 5% |NIV|)": err[tail].abs().mean(),
        "Baseline MAE (MW)": mae_base,
        "MAE skill vs baseline": 1 - mae / mae_base,               # > 0 = beats the training-mean forecast
    }


stats = {}
for run, pred in predictions.items():
    d = run_frame(run)
    y_true = d.loc[pred.index, "niv_mw"]
    y_base = d.loc[d["trading_day"] < split_day, "niv_mw"].mean()   # training-set mean for this run's window
    stats[run] = forecast_stats(y_true, pred, y_base)

forecast_stats_table = pd.DataFrame(stats).T.astype({"periods": int}).round(3)
forecast_stats_table